## 도구 정의

#### 패키지 제공도구

In [1]:
from langchain_tavily import TavilySearch

# TavilySearch: 웹 검색 도구
tavily_search = TavilySearch(
    max_results=5,          # 최대 결과 수
    topic="general",        # "general" | "news" | "finance"
    search_depth="basic",   # "basic" | "advanced"
)


#### 커스텀 도구

In [2]:
from langchain.tools import tool

@tool
def calculator(expression: str) -> str:
    """수학 표현식을 안전하게 계산합니다.

    Args:
        expression: 계산할 수학 표현식 (예: "2 + 3 * 4")
    """
    import ast
    import operator as op

    allowed_ops = {
        ast.Add: op.add, ast.Sub: op.sub,
        ast.Mult: op.mul, ast.Div: op.truediv,
        ast.Pow: op.pow, ast.USub: op.neg,
    }

    def safe_eval(node):
        if isinstance(node, ast.Constant):
            return node.value
        elif isinstance(node, ast.BinOp):
            return allowed_ops[type(node.op)](safe_eval(node.left), safe_eval(node.right))
        elif isinstance(node, ast.UnaryOp):
            return allowed_ops[type(node.op)](safe_eval(node.operand))
        else:
            raise ValueError(f"지원하지 않는 연산: {type(node)}")

    try:
        tree = ast.parse(expression, mode='eval')
        result = safe_eval(tree.body)
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {e}"

# 패키지 도구 + 커스텀 도구를 함께 사용
tools = [tavily_search, calculator]


### 도구 바인딩

In [3]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.1)

# bind_tools: 도구 스펙(JSON Schema)을 LLM에 전달
llm_with_tools = llm.bind_tools(tools)


## ToolNode와 도구 실행

In [4]:
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, END, START

# ToolNode: 도구 호출을 자동으로 처리
tool_node = ToolNode(tools)

# 에이전트 노드: LLM 추론
def agent_node(state: MessagesState) -> dict:
    """LLM이 메시지를 분석하고 다음 행동을 결정"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 라우팅 함수: 도구 호출 여부에 따라 분기
def route_after_agent(state: MessagesState) -> str:
    """마지막 메시지에 도구 호출이 있으면 tools로, 없으면 END로 라우팅"""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

# 그래프 구성
graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", route_after_agent, ["tools", END])
graph.add_edge("tools", "agent")

app = graph.compile()


### 커스텀 도구 실행 노드

In [5]:
from langchain.messages import ToolMessage

def custom_tool_node(state: MessagesState) -> dict:
    """커스텀 도구 실행 노드: 오류 처리 및 결과 로깅 포함"""

    last_message = state["messages"][-1]

    if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
        return {}

    tool_map = {t.name: t for t in tools}
    tool_messages = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        call_id = tool_call["id"]

        if tool_name not in tool_map:
            result = f"오류: '{tool_name}' 도구를 찾을 수 없습니다."
        else:
            try:
                result = tool_map[tool_name].invoke(tool_args)
                print(f"[도구 실행] {tool_name}({tool_args}) -> {str(result)[:100]}...")
            except Exception as e:
                result = f"도구 실행 오류: {e}"

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call_id,
                name=tool_name,
            )
        )

    return {"messages": tool_messages}


### 병렬 도구 호출 (Parallel Tool Calls)

In [ ]:
import asyncio
from langchain.messages import ToolMessage

async def parallel_tool_node(state: MessagesState) -> dict:
    """여러 도구를 비동기로 병렬 실행"""

    last_message = state["messages"][-1]

    if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
        return {}

    tool_map = {t.name: t for t in tools}

    async def run_tool(tool_call: dict) -> ToolMessage:
        """단일 도구를 비동기로 실행"""
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        call_id = tool_call["id"]

        try:
            loop = asyncio.get_event_loop()
            result = await loop.run_in_executor(
                None,
                lambda: tool_map[tool_name].invoke(tool_args)
            )
        except Exception as e:
            result = f"도구 실행 오류: {e}"

        return ToolMessage(
            content=str(result),
            tool_call_id=call_id,
            name=tool_name,
        )

    # 모든 도구 호출을 병렬 실행
    tool_results = await asyncio.gather(
        *[run_tool(tc) for tc in last_message.tool_calls],
        return_exceptions=False,
    )

    return {"messages": list(tool_results)}


## 완전한 실행 예시

In [6]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, MessagesState, END, START
from langgraph.prebuilt import ToolNode

# 1. 도구 정의
tavily_search = TavilySearch(max_results=3, topic="general")

@tool
def calculator(expression: str) -> str:
    """수학 표현식을 계산합니다. 파이썬 수식을 입력받습니다."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"계산 오류: {e}"

tools = [tavily_search, calculator]

# 2. LLM 및 그래프 구성
llm = init_chat_model("openai:gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def agent(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def should_continue(state: MessagesState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

graph = StateGraph(MessagesState)
graph.add_node("agent", agent)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue, ["tools", END])
graph.add_edge("tools", "agent")
app = graph.compile()

# 3. 실행
result = app.invoke({
    "messages": [{"role": "user", "content": "2024년 노벨 물리학상 수상자를 검색하고, 수상자 수를 알려주세요."}]
})

print(result["messages"][-1].content)


2024년 노벨 물리학상은 Geoffrey Hinton과 John Hopfield가 공동 수상했습니다. 이들은 인공지능 발전에 기여한 공로로 선정되었습니다. 따라서 수상자는 총 2명입니다. 

더 자세한 내용은 [여기](https://www.ndtv.com/world-news/what-the-nobel-prize-for-physics-co-winner-fears-6744406)에서 확인할 수 있습니다.
